C:\Users\nickc\AppData\Local\Temp\ipykernel_12152\2139272059.py:34: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["ts"] = pd.to_datetime(df["ts"], errors="coerce")
C:\Users\nickc\AppData\Local\Temp\ipykernel_12152\2139272059.py:196: UserWarning: Glyph 24402 (\N{CJK UNIFIED IDEOGRAPH-5F52}) missing from font(s) DejaVu Sans.
  plt.legend(); plt.tight_layout()
C:\Users\nickc\AppData\Local\Temp\ipykernel_12152\2139272059.py:196: UserWarning: Glyph 19968 (\N{CJK UNIFIED IDEOGRAPH-4E00}) missing from font(s) DejaVu Sans.
  plt.legend(); plt.tight_layout()
C:\Users\nickc\AppData\Local\Temp\ipykernel_12152\2139272059.py:196: UserWarning: Glyph 21270 (\N{CJK UNIFIED IDEOGRAPH-5316}) missing from font(s) DejaVu Sans.
  plt.legend(); plt.tight_layout()
C:\Users\nickc\AppData\Local\Temp\ipykernel_12152\2139272059.py:196: UserWarning: Glyph 36827 (\N{CJK UNI

输出目录： C:\Users\nickc\yanda\btc\eth_analysis0824


FileNotFoundError: 找不到 ahr999_daily.xlsx/.csv

In [2]:
# === ETH：峰值为中心（可用峰前 + 峰后）拟合；动态波动退火 + 加权DTW ===
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

# ========= 基本配置 =========
IN_ETH  = Path("eth_2025.xlsx")
OUTDIR  = Path("eth_analysis0824"); OUTDIR.mkdir(exist_ok=True)

# 锚点仍记录 BTC 减半，仅做参考；真正对齐以“ETH 自己的峰值”为中心
HALVINGS = {2018: pd.Timestamp("2016-07-09"),
            2021: pd.Timestamp("2020-05-11"),
            2025: pd.Timestamp("2024-04-20")}

ETH_PEAKS = {2018: pd.Timestamp("2018-01-13"),
             2021: pd.Timestamp("2021-11-10"),
             2025: pd.Timestamp("2025-08-22")}

# 目标窗口：优先峰前 PRE_TARGET，若不足则用峰后补到 TOTAL_TARGET（再加上 POST_TARGET）
PRE_TARGET   = 300     # 理想的峰前天数
POST_TARGET  = 120     # 理想的峰后天数
TOTAL_TARGET = 420     # 最少需要的总天数（若某轮达不到，会自动降级到三轮的最小可用长度）
MAX_CAP      = 500     # 单轮最多用到的总天数（防过长）

# 动态退火 & 对齐参数
ROLL_WIN       = 28
DTW_BAND       = 0.06
GUARD_END      = 60
PEAK_W_ALPHA   = 3.5   # 强调把“前置波峰/临峰”压齐
LOCAL_WIN_U    = 0.06
VOL_TARGET_MULT= 1.35  # 放大目标波动（便于观察/拟合）
SCALE_CLIP     = (0.4, 3.0)
TAIL_WIN       = 20

EPS = 1e-9

# ========= 工具 =========
def read_two_col_excel(fp: Path) -> pd.DataFrame:
    assert fp.exists(), f"未找到文件：{fp}"
    df_raw = pd.read_excel(fp, engine="openpyxl", header=None)
    if df_raw.shape[0] <= 3 and df_raw.shape[1] > df_raw.shape[0]:
        df_raw = df_raw.T
    df = df_raw.iloc[:, :2].copy()
    df.columns = ["ts","price"]
    df["ts"] = pd.to_datetime(df["ts"], errors="coerce")
    df["price"] = pd.to_numeric(df["price"], errors="coerce")
    return df.dropna(subset=["ts","price"]).drop_duplicates("ts").sort_values("ts")

def to_daily(df: pd.DataFrame) -> pd.Series:
    s = df.set_index("ts")["price"].resample("D").last()
    s = s.reindex(pd.date_range(s.index.min().normalize(), s.index.max().normalize(), freq="D")).ffill()
    s.index.name = "ts"; return s

def pick_peak_in_series(series: pd.Series, peak_dt: pd.Timestamp, tol_days=3) -> pd.Timestamp:
    peak_dt = pd.Timestamp(peak_dt).normalize()
    if peak_dt in series.index: return peak_dt
    seg = series.loc[peak_dt - pd.Timedelta(days=tol_days): peak_dt + pd.Timedelta(days=tol_days)]
    if not seg.empty: return seg.idxmax()
    return series.idxmax()

def window_around_peak(series: pd.Series, peak: pd.Timestamp,
                       pre_target: int, post_target: int, total_target: int, cap:int=500) -> pd.Series:
    """尽量取峰前pre_target、峰后post_target；若峰前不足，用峰后补齐到total_target；总长度≤cap。"""
    peak = pd.Timestamp(peak).normalize()
    series = series.asfreq("D").ffill()
    # 实际可用
    pre_avail  = (peak - max(series.index.min(), peak - pd.Timedelta(days=cap))).days
    post_avail = (min(series.index.max(), peak + pd.Timedelta(days=cap)) - peak).days
    pre_use  = min(pre_target, pre_avail)
    post_use = min(post_target, post_avail)
    # 如果总长度不够 total_target，就从峰后再补
    need_more = max(0, total_target - (pre_use + post_use))
    add_post  = min(need_more, post_avail - post_use)
    post_use += max(0, add_post)
    # 最终切片
    lo = peak - pd.Timedelta(days=pre_use)
    hi = peak + pd.Timedelta(days=post_use)
    return series.loc[lo:hi]

def roll_stats(x, win=ROLL_WIN):
    z = np.log2(np.maximum(x, EPS))
    s = pd.Series(z)
    mu = s.rolling(win, center=True, min_periods=win//2).mean().bfill().ffill().values
    sd = s.rolling(win, center=True, min_periods=win//2).std(ddof=0).bfill().ffill().values
    sd = np.where(sd==0, np.nan, sd)
    sd = pd.Series(sd).fillna(np.nanmedian(sd) if np.isfinite(np.nanmedian(sd)) else 1.0).values
    return z, mu, sd

def peak_weights(z_ref):
    # 峰附近&临峰斜率变化大，给更高权重
    g = np.gradient(z_ref)
    p = np.abs(g); p /= (np.percentile(p, 95) + 1e-9)
    # 同时在 u∈[0.7,1.1] 内（峰附近）再加一点钟形权
    n = len(z_ref); u = np.linspace(0,1,n)
    bell = np.exp(-((u-0.95)/0.15)**2)
    w = 1.0 + PEAK_W_ALPHA * np.clip(p, 0, 1) + 1.5 * bell
    return w

def dtw_weighted(x, y, w, band, guard_end=0):
    n, m = len(x), len(y)
    m_eff = max(10, m-guard_end); band = max(1, int(band*m_eff)); INF=1e18
    D = np.full((n+1, m_eff+1), INF); D[0,0]=0.0
    for i in range(1,n+1):
        j0, j1 = max(1, i-band), min(m_eff, i+band)
        xi = x[i-1]
        for j in range(j0, j1+1):
            d = w[j-1]*(xi - y[j-1])**2
            D[i,j] = d + min(D[i-1,j], D[i,j-1], D[i-1,j-1])
    i,j=n,m_eff; pi,pj=[],[]
    while i>0 and j>0:
        pi.append(i-1); pj.append(j-1)
        i,j = min(((i-1,j),(i,j-1),(i-1,j-1)), key=lambda t: D[t])
    return np.array(pi[::-1]), np.array(pj[::-1]), D[n,m_eff]

def u_map_per_index(pi, pj, n_src, m_ref):
    buckets = [[] for _ in range(n_src)]
    for ii, jj in zip(pi, pj): buckets[ii].append(jj)
    j_mean = np.array([np.mean(b) if b else np.nan for b in buckets], dtype=float)
    j_mean = pd.Series(j_mean).interpolate().bfill().ffill().values
    return j_mean / (m_ref - 1)

def amplitude_rescale(Z, MU, SD, u_map, u_ref, sigma_target):
    sig_u = np.interp(u_map, u_ref, sigma_target)
    scale = np.clip(sig_u / SD, SCALE_CLIP[0], SCALE_CLIP[1])
    Z_new = MU + (Z - MU) * scale
    return Z_new, scale

def local_ab(u_obs, Z_ref_at_pj, Z_obs_at_pi, u_grid, win=LOCAL_WIN_U):
    a = np.full_like(u_grid, np.nan, dtype=float)
    b = np.full_like(u_grid, np.nan, dtype=float)
    for k, ug in enumerate(u_grid):
        mask = (np.abs(u_obs - ug) <= win)
        if mask.sum() >= 12:
            X = np.vstack([np.ones(mask.sum()), Z_ref_at_pj[mask]]).T
            y = Z_obs_at_pi[mask]
            aa, bb = np.linalg.lstsq(X, y, rcond=None)[0]
            a[k], b[k] = aa, bb
    a = pd.Series(a).interpolate().bfill().ffill().rolling(11, center=True, min_periods=3).mean().bfill().ffill().values
    b = pd.Series(b).interpolate().bfill().ffill().rolling(11, center=True, min_periods=3).mean().bfill().ffill().values
    return a, b

# ========= 读入 ETH，构造“峰中心窗口”（可用峰前 + 峰后）=========
eth = to_daily(read_two_col_excel(IN_ETH)); assert not eth.empty, "ETH 数据为空"

segments, peaks_used = {}, {}
for cyc in [2018, 2021, 2025]:
    pk = pick_peak_in_series(eth, ETH_PEAKS[cyc], tol_days=3)
    peaks_used[cyc] = pk
    seg = window_around_peak(eth, pk, PRE_TARGET, POST_TARGET, TOTAL_TARGET, cap=MAX_CAP)
    segments[cyc] = seg

# 三段统一长度：取三者最小长度（避免长度不等），最低 120 天兜底
min_len = min(len(s) for s in segments.values())
M = max(120, min(min_len, MAX_CAP))
for cyc in segments:
    # 居中切（保证峰在序列中心附近）
    s = segments[cyc]
    if len(s) > M:
        extra = len(s) - M
        drop_left = extra // 2
        segments[cyc] = s.iloc[drop_left: drop_left + M]
    else:
        segments[cyc] = s  # 已是最短

# 计算“相对峰日 t”和统一 u∈[0,1]
t_grid = {}
for cyc, s in segments.items():
    idx = s.index
    t = (idx - peaks_used[cyc]).days.astype(int)      # t<0 峰前，t>=0 峰后
    t_grid[cyc] = pd.Series(t, index=idx)
# 用每段自身的 t 线性映射到 u（保证峰在 u≈0.5~0.6 一带，取决于峰前/后比例）
u_series = {}
for cyc, s in segments.items():
    t = t_grid[cyc].values
    u = (t - t.min()) / (t.max() - t.min() + 1e-9)
    u_series[cyc] = pd.Series(u, index=s.index)

# 转 log2 & 滚动统计
Z, MU, SD = {}, {}, {}
for cyc, s in segments.items():
    Z[cyc], MU[cyc], SD[cyc] = roll_stats(s.values, win=ROLL_WIN)

# 目标波动曲线 sigma*(u)：用 2018/2021 的 sd 在各自 u 轴上的均值 + 单调递减 + 放大
u_ref = np.linspace(0, 1, M)
def to_u_series(sd, u_obs):
    df = pd.DataFrame({"u": u_obs.values, "sd": sd}).sort_values("u")
    sd_smooth = df["sd"].rolling(11, center=True, min_periods=3).mean()
    return np.interp(u_ref, df["u"].values, sd_smooth.fillna(method="bfill").fillna(method="ffill").values)

base_sd = []
for cyc in (2018, 2021):
    base_sd.append(to_u_series(SD[cyc], u_series[cyc]))
sigma_target = np.nanmean(np.vstack(base_sd), axis=0)

# 单调递减 + 放大（保持 Series 再取 values）
s = pd.Series(sigma_target).bfill().ffill()              # 补 NaN
s = s.iloc[::-1].cummin().iloc[::-1]                     # 从右到左累积最小
sigma_target = (s.values * VOL_TARGET_MULT)


# 第1轮退火（按各自 u）
def rescale_by_u(cyc):
    return amplitude_rescale(Z[cyc], MU[cyc], SD[cyc], u_series[cyc].values, u_ref, sigma_target)[0]
Z2018_1 = Z[2018]
Z2021_1 = rescale_by_u(2021)
Z2025_1 = rescale_by_u(2025)

# 对齐到 2018 作为参考
def align_once(Zref, Z21_cur, Z25_cur):
    def zscore_from(Zv):
        s = pd.Series(Zv)
        mu = s.rolling(ROLL_WIN, center=True, min_periods=ROLL_WIN//2).mean().bfill().ffill()
        sd = s.rolling(ROLL_WIN, center=True, min_periods=ROLL_WIN//2).std(ddof=0).bfill().ffill()
        sd = sd.replace(0, sd.median() or 1.0)
        return ((s-mu)/sd).values
    zref = zscore_from(Zref)
    z21  = zscore_from(Z21_cur)
    z25  = zscore_from(Z25_cur)
    wref = peak_weights(zref)
    pi21, pj21, _ = dtw_weighted(z21, zref, wref, DTW_BAND, GUARD_END)
    pi25, pj25, _ = dtw_weighted(z25, zref, wref, DTW_BAND, GUARD_END)
    return (pi21, pj21), (pi25, pj25)

(PI21_1, PJ21_1), (PI25_1, PJ25_1) = align_once(Z2018_1, Z2021_1, Z2025_1)

# 路径→每点 u 映射（把 2018 的“参考 u_ref”映到另外两段）
def per_index_map(pi, pj):
    return u_map_per_index(pi, pj, n_src=M, m_ref=M)  # 输出长度=M，范围≈[0,1]
u21_map = per_index_map(PI21_1, PJ21_1)
u25_map = per_index_map(PI25_1, PJ25_1)

# 第2轮退火，再对齐
Z2021_2, scale21 = amplitude_rescale(Z[2021], MU[2021], SD[2021], u21_map, u_ref, sigma_target)
Z2025_2, scale25 = amplitude_rescale(Z[2025], MU[2025], SD[2025], u25_map, u_ref, sigma_target)
(PI21, PJ21), (PI25, PJ25) = align_once(Z[2018], Z2021_2, Z2025_2)

# 局部回归 a(u), b(u)：2018 为参考，不混合
u_grid = np.linspace(0,1,M)
def local_fit(pi, pj, Z_src, Z_ref):
    # 取配对点
    Zr = Z_ref[pj]; Zs = Z_src[pi]
    # 反推出 src 在“参考 u”上的 u 座标
    u_obs = pj/(M-1)
    return local_ab(u_obs, Zr, Zs, u_grid, win=LOCAL_WIN_U)

a21, b21 = local_fit(PI21, PJ21, Z2021_2, Z[2018])
a25, b25 = local_fit(PI25, PJ25, Z2025_2, Z[2018])
# 轻量单调：2018 ≥ 2021 ≥ 2025
b21 = np.minimum(b21, 1.00 - 0.02)
b25 = np.minimum(b25, b21 - 0.02)

# 还原幅度（2^Z），观察三段重叠
y18_ref = 2**Z[2018]
y21_proj = 2**(a21 + b21 * Z[2018])
y25_proj = 2**(a25 + b25 * Z[2018])

# ========= 输出图 =========
plt.figure(figsize=(12.6,4.9))
plt.plot(u_grid, y18_ref, label="2018（参考：峰中心窗口）")
plt.plot(u_grid, y21_proj, label="2021（动态退火对齐）")
plt.plot(u_grid, y25_proj, label="2025（动态退火对齐）")
plt.axvline(0.5, ls="--", c="gray", lw=1)  # 峰大致落在 u≈0.5 附近
plt.title(f"ETH 峰中心对齐｜动态波动退火 + 加权DTW（VOL×{VOL_TARGET_MULT}, 带宽={DTW_BAND}）")
plt.xlabel("统一进度 u（0=窗口起点，~0.5≈峰，1=窗口终点）")
plt.ylabel("AHR风格幅度（对数还原）")
plt.legend(); plt.tight_layout()
plt.savefig(OUTDIR / "eth_peak_centric_overlaps.png", dpi=170); plt.close()

# ========= 导出指标 & 合并 =========
def corr_overlap(a, b):
    ok = (~np.isnan(a)) & (~np.isnan(b))
    if ok.sum() < 20: return np.nan
    A, B = a[ok], b[ok]
    return float(np.corrcoef(A, B)[0,1])

metrics = {
    "M_window": M,
    "dtw_band": DTW_BAND,
    "peak_w_alpha": PEAK_W_ALPHA,
    "vol_target_mult": VOL_TARGET_MULT,
    "scale21_median": float(np.median(scale21)),
    "scale25_median": float(np.median(scale25)),
    "corr_21_vs_18": corr_overlap(y21_proj, y18_ref),
    "corr_25_vs_18": corr_overlap(y25_proj, y18_ref),
    "peak_used_2018": peaks_used[2018].date(),
    "peak_used_2021": peaks_used[2021].date(),
    "peak_used_2025": peaks_used[2025].date(),
}
pd.DataFrame(list(metrics.items()), columns=["item","value"]).to_csv(
    OUTDIR / "eth_peak_centric_metrics.csv", index=False, encoding="utf-8-sig"
)

pd.DataFrame({"u": u_grid,
              "amp_2018_ref": y18_ref,
              "amp_2021_proj": y21_proj,
              "amp_2025_proj": y25_proj}).to_csv(
    OUTDIR / "eth_peak_centric_overlaps_on_u.csv", index=False, encoding="utf-8-sig"
)

print("输出：")
print("  图：", (OUTDIR / "eth_peak_centric_overlaps.png").resolve())
print("  指标：", (OUTDIR / "eth_peak_centric_metrics.csv").resolve())
print("  合并：", (OUTDIR / "eth_peak_centric_overlaps_on_u.csv").resolve())


C:\Users\nickc\AppData\Local\Temp\ipykernel_12684\965366163.py:45: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["ts"] = pd.to_datetime(df["ts"], errors="coerce")
C:\Users\nickc\AppData\Local\Temp\ipykernel_12684\965366163.py:188: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  return np.interp(u_ref, df["u"].values, sd_smooth.fillna(method="bfill").fillna(method="ffill").values)
C:\Users\nickc\AppData\Local\Temp\ipykernel_12684\965366163.py:266: UserWarning: Glyph 32479 (\N{CJK UNIFIED IDEOGRAPH-7EDF}) missing from font(s) DejaVu Sans.
  plt.legend(); plt.tight_layout()
C:\Users\nickc\AppData\Local\Temp\ipykernel_12684\965366163.py:266: UserWarning: Glyph 19968 (\N{CJK UNIFIED IDEOGRAPH-4E00}) missing from font(s) DejaVu Sans.
  plt.legend(); plt.tight_layout

输出：
  图： C:\Users\nickc\yanda\btc\eth_analysis0824\eth_peak_centric_overlaps.png
  指标： C:\Users\nickc\yanda\btc\eth_analysis0824\eth_peak_centric_metrics.csv
  合并： C:\Users\nickc\yanda\btc\eth_analysis0824\eth_peak_centric_overlaps_on_u.csv
